# 7장. RAG 시스템 평가 — 02. RAGAS로 RAG 시스템 평가하기

## 학습 목표

- RAGAS의 4가지 핵심 지표(Context Precision, Context Recall, Faithfulness, Answer Relevancy)를 이해
- 앞 노트북에서 생성한 합성 테스트셋으로 RAG 시스템을 평가
- 지표별 점수를 해석하고 개선 방향을 도출

## 사전 지식

- `01-Test-Dataset-Generator-RAGAS.ipynb` 완료 (테스트셋 `data/attention_synthetic_testset.csv` 생성)
- RAG 시스템 구조 이해 (검색기 + 생성기)

## RAGAS 4가지 지표 한눈에 보기

```mermaid
flowchart TD
    Q[❓ 질문] --> R[Retriever<br/>검색기]
    R --> C[검색된 컨텍스트]
    Q --> G[Generator<br/>생성기]
    C --> G
    G --> A[생성된 답변]
    GT[ground_truth<br/>정답] -.-> M1
    GT -.-> M2

    C --> M1[Context Precision<br/>관련 문서가<br/>상위에 있는가?]
    C --> M2[Context Recall<br/>필요한 정보를<br/>모두 찾았는가?]
    A --> M3[Faithfulness<br/>답변이 컨텍스트에<br/>근거하는가?]
    Q --> M4[Answer Relevancy<br/>답변이 질문과<br/>관련 있는가?]
    A --> M3
    A --> M4

    M1 --> S[종합 평가 점수]
    M2 --> S
    M3 --> S
    M4 --> S

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class Q,GT input
    class R,G process
    class C,A process
    class M1,M2,M3,M4 storage
    class S output
```

> **검색(Retrieval) 평가**: Context Precision + Context Recall
> **생성(Generation) 평가**: Faithfulness + Answer Relevancy

---

## 환경 설정

In [ ]:
# 필요한 패키지 설치
# !pip install -qU langchain==0.2.16 ragas==0.1.19 faiss-cpu


In [ ]:
from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()


---

## 테스트 데이터셋 로드

`01-Test-Dataset-Generator-RAGAS.ipynb`에서 생성한 합성 테스트셋을 불러옵니다.

In [ ]:
import os

# 데이터 파일 경로 확인
for _path in ["data/attention_synthetic_testset.csv", "data/attention_paper.pdf"]:
    if not os.path.exists(_path):
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {_path}\n현재 디렉토리: {os.getcwd()}")

In [ ]:
import pandas as pd
from datasets import Dataset

# CSV 파일 로드
df = pd.read_csv("data/attention_synthetic_testset.csv")

print(f"테스트 케이스 수: {len(df)}")
print(f"컬럼: {list(df.columns)}")

df.head()


### contexts 컬럼 변환

CSV로 저장하면 리스트가 문자열로 저장돼요. `ast.literal_eval()`로 다시 리스트로 변환합니다.

In [ ]:
# ---------------------------------------------------
# contexts 컬럼: CSV에서 문자열로 저장된 리스트를 실제 리스트로 복원
# ---------------------------------------------------
import ast

# Dataset으로 변환
# Dataset: HuggingFace datasets 라이브러리의 메모리 내 데이터셋
test_dataset = Dataset.from_pandas(df)

# contexts 컬럼을 문자열에서 리스트로 변환
def convert_to_list(example):
    """문자열로 저장된 리스트를 실제 리스트로 변환"""
    contexts = ast.literal_eval(example["contexts"])
    return {"contexts": contexts}

test_dataset = test_dataset.map(convert_to_list)

print(f"Dataset: {test_dataset}")
print(f"\n첫 번째 케이스의 contexts:")
print(test_dataset[0]["contexts"])

---

## RAG 시스템 구축

평가 대상이 되는 RAG 시스템을 구축합니다. Attention 논문을 기반으로 질문에 답변하는 시스템이에요.

In [ ]:
# ---------------------------------------------------
# RAG 시스템 구축 (문서 로드 → 분할 → 벡터화 → 체인 조립)
# ---------------------------------------------------
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 1단계: 문서 로드
loader = PyMuPDFLoader("data/attention_paper.pdf")
docs = loader.load()
print(f"문서 로드 완료: {len(docs)} 페이지")

# 2단계: 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50
)
split_documents = text_splitter.split_documents(docs)
print(f"문서 분할 완료: {len(split_documents)} 청크")

# 3단계: 임베딩 및 벡터 DB 생성
# FAISS: 메타(Facebook AI)에서 만든 고속 유사도 검색 라이브러리
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(
    documents=split_documents,
    embedding=embeddings
)
print("벡터 DB 생성 완료")

# 4단계: Retriever 생성
# as_retriever(): 벡터 DB를 LangChain Retriever 인터페이스로 변환
retriever = vectorstore.as_retriever()

# 5단계: 프롬프트 정의
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks about the Transformer architecture.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.

#Context:
{context}

#Question:
{question}

#Answer:"""
)

# 6단계: LLM 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 7단계: RAG 체인 생성
# RunnablePassthrough(): 입력을 그대로 다음 단계로 전달
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG 시스템 구축 완료")

### RAG 시스템 동작 확인

In [ ]:
# 테스트 질문으로 RAG 시스템 확인
test_question = "What is the main architecture proposed in this paper?"
answer = chain.invoke(test_question)

print("=" * 80)
print("❓ 질문:", test_question)
print("=" * 80)
print("💬 답변:")
print(answer)
print("=" * 80)


---

## 테스트 데이터셋에 대한 답변 생성

RAGAS 평가에는 RAG 시스템이 생성한 `answer`가 필요해요. 테스트셋의 모든 질문에 답변을 생성합니다.

In [ ]:
# 배치 데이터셋 생성
batch_questions = [question for question in test_dataset["question"]]

print(f"총 {len(batch_questions)}개의 질문에 대해 답변 생성")
print(f"\n질문 예시:")
for i, q in enumerate(batch_questions[:3], 1):
    print(f"{i}. {q}")


In [ ]:
# ---------------------------------------------------
# 테스트셋 전체 질문에 대해 RAG 답변 배치 생성
# ---------------------------------------------------
# ============================================================
# TODO: chain.batch()로 모든 질문에 답변을 생성하세요
# 힌트: answers = chain.batch(batch_questions)
# 예상 결과: len(answers) == len(batch_questions)
# ============================================================



### 답변을 데이터셋에 추가

In [ ]:
# 'answer' 컬럼이 이미 있으면 제거하고 새로 추가
if "answer" in test_dataset.column_names:
    test_dataset = test_dataset.remove_columns(["answer"])

test_dataset = test_dataset.add_column("answer", answers)

print("Dataset에 answer 컬럼 추가 완료")
print(f"컬럼: {test_dataset.column_names}")


---

## RAGAS 4가지 지표 상세 이해

평가를 실행하기 전에 각 지표가 정확히 무엇을 측정하는지 살펴볼게요.

### 지표 1. Context Precision (컨텍스트 정확도)

**검색기(Retriever)가 관련 문서를 상위에 올려두었는가?**

이상적으로는 가장 관련성 높은 문서가 1번에 오고, 덜 관련된 문서가 뒤에 와야 해요. Reranker가 없다면 이 지표가 낮게 나오는 경우가 많습니다.

**계산 방법**:

$$\text{Context Precision@K} = \frac{\sum_{k=1}^{K} (\text{Precision@k} \times v_k)}{\text{Total relevant items in top K}}$$

- `v_k` = k번째 문서의 관련성 (0 또는 1)
- 높을수록 관련 문서가 상위에 잘 배치됨

**목표 점수**: > 0.8

**점수가 낮을 때**: 임베딩 모델 개선, 청크 크기 최적화, Reranker 추가

### 지표 2. Context Recall (컨텍스트 재현율)

**정답을 생성하는 데 필요한 모든 정보가 검색되었는가?**

`ground_truth`의 각 주장(claim)이 검색된 컨텍스트에서 확인 가능한지 검사해요. 검색 범위(k)가 좁으면 이 지표가 낮게 나옵니다.

**계산 방법**:

$$\text{Context Recall} = \frac{|\text{정답의 주장 중 컨텍스트로 설명 가능한 것}|}{|\text{정답의 전체 주장}|}$$

**목표 점수**: > 0.85

**점수가 낮을 때**: 검색 문서 수(k) 늘리기, 하이브리드 검색(키워드 + 벡터) 적용

### 지표 3. Faithfulness (충실도)

**생성된 답변이 컨텍스트에만 근거하는가? — 할루시네이션(Hallucination) 체크**

LLM이 컨텍스트에 없는 내용을 꾸며내는 것을 Hallucination이라고 해요. Faithfulness는 이를 정량화합니다.

**계산 방법**:

$$\text{Faithfulness} = \frac{|\text{답변의 주장 중 컨텍스트로 뒷받침되는 것}|}{|\text{답변의 전체 주장}|}$$

**예시**:
- 컨텍스트: "Transformer는 Self-Attention을 사용합니다."
- 높은 충실도 답변: "핵심은 Self-Attention입니다." (컨텍스트 내)
- 낮은 충실도 답변: "CNN과 Self-Attention을 사용합니다." (CNN은 컨텍스트에 없음)

**목표 점수**: > 0.9

**점수가 낮을 때**: 프롬프트에 "컨텍스트 기반으로만 답변" 강조, temperature 낮추기, 더 강력한 LLM 사용

### 지표 4. Answer Relevancy (답변 관련성)

**생성된 답변이 질문에 직접적으로 답하는가?**

답변으로부터 역으로 질문을 생성한 뒤, 원래 질문과의 임베딩 유사도를 측정해요. 질문과 무관한 내용이 많을수록 낮아집니다.

**계산 방법**:

$$\text{Answer Relevancy} = \frac{1}{N} \sum_{i=1}^N \cos(E_{\text{생성질문}_i}, E_{\text{원래질문}})$$

**점수가 낮은 이유**: 질문에 엉뚱한 답변, 불필요한 정보 과다 포함, 너무 일반적인 답변

**목표 점수**: > 0.85

**점수가 낮을 때**: 프롬프트 개선, Few-shot 예제 추가

---

## RAGAS 평가 실행

In [ ]:
# ---------------------------------------------------
# RAGAS 4가지 지표 평가 실행
# ---------------------------------------------------
# ============================================================
# TODO: evaluate() 함수를 호출해 RAGAS 평가를 실행하세요
# 힌트: evaluate(dataset=test_dataset, metrics=[context_precision, context_recall, faithfulness, answer_relevancy])
# 예상 결과: 각 메트릭 점수가 담긴 result 객체 반환
# ============================================================

from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)



---

## 평가 결과 분석

In [ ]:
# DataFrame으로 변환
result_df = result.to_pandas()

print(f"평가 결과: {len(result_df)} rows")
print(f"컬럼: {list(result_df.columns)}")

result_df.head()


### 지표별 점수 분석

> **해석 기준**: 0.8 이상이면 양호, 0.6~0.8이면 개선 필요, 0.6 미만이면 문제 있음

In [ ]:
# 4가지 메트릭 점수만 추출
metric_cols = ['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']
metrics_df = result_df[metric_cols]

print("=" * 80)
print("📊 RAGAS 평가 결과")
print("=" * 80)

# 각 메트릭의 평균 점수
print("\n평균 점수:")
for metric in metric_cols:
    score = metrics_df[metric].mean()
    status = "✅" if score > 0.8 else "⚠️" if score > 0.6 else "❌"
    print(f"{status} {metric:25s}: {score:.3f}")

# 전체 평균
overall_score = metrics_df.mean().mean()
print(f"\n{'=' * 80}")
print(f"🎯 전체 평균 점수: {overall_score:.3f}")
print(f"{'=' * 80}")

metrics_df


### 점수 분포 확인

In [ ]:
# 각 메트릭의 통계 정보
print("=" * 80)
print("📈 메트릭별 통계")
print("=" * 80)

metrics_df.describe().round(3)


### 점수가 낮은 케이스 분석

어떤 질문에서 점수가 낮게 나왔는지 확인하고 개선 방향을 찾아볼게요.

In [ ]:
# 각 메트릭별로 가장 낮은 점수를 받은 케이스 찾기
print("=" * 80)
print("⚠️ 개선이 필요한 케이스")
print("=" * 80)

for metric in metric_cols:
    # 가장 낮은 점수의 인덱스
    worst_idx = result_df[metric].idxmin()
    worst_score = result_df.loc[worst_idx, metric]
    
    print(f"\n{metric}이 가장 낮은 케이스 (점수: {worst_score:.3f}):")
    print(f"질문: {result_df.loc[worst_idx, 'question'][:100]}...")
    print(f"답변: {result_df.loc[worst_idx, 'answer'][:100]}...")


---

## 평가 결과 저장

In [ ]:
# CSV 파일로 저장
output_path = "data/attention_evaluation_result.csv"
result_df.to_csv(output_path, index=False)

print(f"✅ 평가 결과가 저장되었습니다: {output_path}")
print(f"총 {len(result_df)}개의 평가 케이스")


---

## 정리

### RAGAS 4가지 지표 요약

| 지표 | 측정 대상 | 낮을 때 원인 | 개선 방법 |
|------|----------|------------|---------|
| **Context Precision** | 관련 문서의 상위 배치 | 임베딩 품질 부족, Reranker 없음 | 임베딩 모델 개선, Reranker 추가 |
| **Context Recall** | 필요 정보의 검색 완성도 | k 값 작음, 청크 분리 | k 증가, 하이브리드 검색 |
| **Faithfulness** | 답변의 컨텍스트 근거율 | Hallucination 발생 | 프롬프트 강화, temperature 0 |
| **Answer Relevancy** | 답변과 질문의 관련성 | 엉뚱한 답변, 과도한 부가 내용 | 프롬프트 개선, Few-shot 추가 |

### RAGAS vs LangSmith 비교

RAGAS는 로컬에서 빠르게 RAG 파이프라인을 평가할 때 유용해요. 다음 노트북에서 배울 LangSmith는 프로덕션 환경에서의 지속적 모니터링에 특화되어 있습니다.

| 특징 | RAGAS | LangSmith |
|------|-------|-----------|
| 초점 | RAG 특화 4가지 지표 | 범용 평가 + 모니터링 |
| 데이터 관리 | 로컬 CSV | 클라우드 데이터셋 |
| 시각화 | Python 코드 | 웹 대시보드 |
| 실시간 추적 | 없음 | 있음 |

### 다음 노트북 예고

다음에는 **LangSmith**를 활용한 평가 방법을 배워요. 데이터셋을 클라우드에 저장하고, LLM 자체를 평가자로 쓰는 **LLM-as-Judge** 패턴, 그리고 임베딩 거리 기반 평가를 다뤄볼게요.